# AGROBRIDGE Pomegranate Dataset Audit

Cloud-only: datasets live in the temporary Kaggle/Colab runtime. Nothing heavy belongs on the developer laptop or GitHub.

Audit gates: source/license, class distribution, corrupt files, dimensions, exact duplicates, and leakage-safe split preparation.

In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter
import hashlib

DATA_ROOT=Path('/kaggle/input')
WORK_ROOT=Path('/kaggle/working/agrobridge_pomegranate_audit')
WORK_ROOT.mkdir(parents=True,exist_ok=True)

def audit(root):
    exts={'.jpg','.jpeg','.png','.webp'}; files=[p for p in Path(root).rglob('*') if p.suffix.lower() in exts]
    bad=[]; sizes=Counter(); hashes=Counter()
    for p in files:
        try:
            with Image.open(p) as im: im.verify()
            with Image.open(p) as im: sizes[im.size]+=1
            hashes[hashlib.sha256(p.read_bytes()).hexdigest()]+=1
        except Exception as e: bad.append((str(p),str(e)))
    return {'images':len(files),'corrupt':bad,'dimensions':sizes,'exact_duplicates':sum(v-1 for v in hashes.values() if v>1)}

print('Mounted cloud datasets:', [p.name for p in DATA_ROOT.iterdir()] if DATA_ROOT.exists() else [])
print('Set DATASET_DIR to an approved mounted dataset before running audit.')

## Execution
1. Attach an approved dataset in Kaggle/Colab.
2. Set `DATASET_DIR` to that mounted directory.
3. Run `audit(DATASET_DIR)`.
4. Review duplicates/corrupt files before training.
5. Never commit raw images or generated copies to this repository.